# Export Seurat Components from scRNA-seq data

This script reads the Seurat `.rds` file and exports the major components (counts matrix, cell metadata, embeddings) into `.mtx` and `.csv` files. Downstream these exported files will be used to construct an AnnData object that will be used for label transfer with Xenium data. Here I only exported the raw counts layer.

In [1]:
library(Seurat)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t




### Read data

In [2]:
setwd("/home/workspace/temp/artis/")
getwd()

[1] "/home/workspace/temp/artis"

In [3]:
file.info("annotated_integrated_seurat.rds")$size

[1] 9340342771

In [4]:
seurat_obj <- readRDS(gzfile("annotated_integrated_seurat.rds"))

In [5]:
seurat_obj

An object of class Seurat 
47995 features across 145986 samples within 2 assays 
Active assay: RNA (25424 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: SCT
 3 dimensional reductions calculated: pca, integrated.rpca, umap

## Export Seurat slots as csv

In [6]:
setwd("/home/workspace/private/projects/artis/data/seurat")
getwd()

[1] "/home/workspace/private/projects/artis/data/seurat"

### Counts

In [7]:
library(Matrix)

In [ ]:
# Get the sparse matrices (DO NOT convert to dense)
counts_sparse <- GetAssayData(seurat_obj, assay = "RNA", layer = "counts")
#normalized_sparse <- GetAssayData(seurat_obj, assay = "RNA", layer = "data")
#scaled_sparse <- GetAssayData(seurat_obj, assay = "RNA", layer = "scale.data")

# Save as Matrix Market format (.mtx) (Keeps everything sparse)
writeMM(counts_sparse, "RNA_counts.mtx")
#writeMM(normalized_sparse, "RNA_normalized.mtx") # Error here due to memory, but just need raw counts
#writeMM(scaled_sparse, "RNA_scaled.mtx")

In [12]:
# Save gene (row) names and cell (column) names separately
write.csv(data.frame(gene = rownames(counts_sparse)), "genes.csv", quote = FALSE, row.names = FALSE)
write.csv(data.frame(obs = colnames(counts_sparse)), "cells.csv", quote = FALSE, row.names = FALSE)

### Metadata

In [18]:
write.csv(seurat_obj@meta.data, "cell_metadata.csv", quote = TRUE, row.names = TRUE)

### Embeddings

In [19]:
# Export PCA coordinates
write.csv(Embeddings(seurat_obj, reduction = "pca"), "PCA_embeddings.csv", quote = FALSE)

# Export UMAP coordinates
write.csv(Embeddings(seurat_obj, reduction = "umap"), "UMAP_embeddings.csv", quote = FALSE)

# Export Integrated RPCA coordinates
write.csv(Embeddings(seurat_obj, reduction = "integrated.rpca"), "RPCA_embeddings.csv", quote = FALSE)

### Variable features

In [22]:
write.csv(VariableFeatures(seurat_obj), "variable_features.csv", quote = FALSE, row.names = FALSE)

### SCT Assay

In [ ]:
# Export SCT counts
#writeMM(as(GetAssayData(seurat_obj, assay = "SCT", slot = "counts"), "CsparseMatrix"), "SCT_counts.mtx")
#writeMM(as(GetAssayData(seurat_obj, assay = "SCT", slot = "data"), "CsparseMatrix"), "SCT_normalized.mtx")
#writeMM(as(GetAssayData(seurat_obj, assay = "SCT", slot = "scale.data"), "CsparseMatrix"), "SCT_scaled.mtx")